# Exercise File 1 — Interview Must Revision
**Topics Covered:** Missing Values · Business Days · Bigrams · Min/Max without built-ins · 
Min-Max Normalisation · Impute Median by Group · Repeated Category Purchase · Ride Duration / Timedelta · 
IQR Outlier Removal · GroupBy + FFill

Work through each question on your own first, then reveal the optimised solution.

---
## Q1 — Handle Missing Values (Easy)

**Problem:** Given a DataFrame of employee records, fill numeric `salary` gaps with the column mean, and drop rows where `department` is null.

**Sample Input:**

| emp_id | name  | department | salary |
|--------|-------|------------|--------|
| 1      | Alice | Sales      | 70000  |
| 2      | Bob   | NaN        | 80000  |
| 3      | Carol | HR         | NaN    |
| 4      | Dave  | IT         | 90000  |
| 5      | Eve   | Sales      | NaN    |

**Sample Output:**

| emp_id | name  | department | salary |
|--------|-------|------------|--------|
| 1      | Alice | Sales      | 70000  |
| 3      | Carol | HR         | 80000  |
| 4      | Dave  | IT         | 90000  |
| 5      | Eve   | Sales      | 80000  |

> Bob (row 2) dropped — `department` is NaN. Carol & Eve salary filled with mean (70000+80000+90000)/3 = 80000.

In [11]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    'emp_id':     [1, 2, 3, 4, 5],
    'name':       ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'department': ['Sales', None, 'HR', 'IT', 'Sales'],
    'salary':     [70000, 80000, np.nan, 90000, np.nan]
})
print(employees)

df = employees.copy()

df_cleaned = df.dropna(subset='department')

df_cleaned = df_cleaned.fillna(value={'salary':df_cleaned['salary'].mean()})

df_cleaned



   emp_id   name department   salary
0       1  Alice      Sales  70000.0
1       2    Bob        NaN  80000.0
2       3  Carol         HR      NaN
3       4   Dave         IT  90000.0
4       5    Eve      Sales      NaN


,emp_id,name,department,salary
0,1,Alice,Sales,70000.0
2,3,Carol,HR,80000.0
3,4,Dave,IT,90000.0
4,5,Eve,Sales,80000.0


**Concepts to use:**
1. `fillna(df['col'].mean())` — fill numeric NaNs with mean.
2. `dropna(subset=['col'])` — drop rows where a specific column is null.
3. Order matters: fill first, then drop (or vice versa depending on the column).

In [12]:
# Optimised Solution
def clean_employees(df):
    df = df.copy()
    df['salary'] = df['salary'].fillna(df['salary'].mean())
    df = df.dropna(subset=['department'])
    return df

print(clean_employees(employees))


   emp_id   name department   salary
0       1  Alice      Sales  70000.0
2       3  Carol         HR  80000.0
3       4   Dave         IT  90000.0
4       5    Eve      Sales  80000.0


---
## Q2 — Business Days Between Two Dates (Easy)

**Problem:** Given a DataFrame with `start_date` and `end_date`, compute the number of business days (Mon–Fri, excluding weekends) for each row.

**Sample Input:**

| project | start_date | end_date   |
|---------|------------|------------|
| Alpha   | 2024-01-01 | 2024-01-10 |
| Beta    | 2024-03-15 | 2024-03-20 |
| Gamma   | 2024-12-23 | 2024-12-31 |

**Sample Output:**

| project | start_date | end_date   | business_days |
|---------|------------|------------|---------------|
| Alpha   | 2024-01-01 | 2024-01-10 | 7             |
| Beta    | 2024-03-15 | 2024-03-20 | 4             |
| Gamma   | 2024-12-23 | 2024-12-31 | 6             |

In [ ]:
import pandas as pd
import numpy as np

projects = pd.DataFrame({
    'project':    ['Alpha', 'Beta', 'Gamma'],
    'start_date': pd.to_datetime(['2024-01-01', '2024-03-15', '2024-12-23']),
    'end_date':   pd.to_datetime(['2024-01-10', '2024-03-20', '2024-12-31'])
})
print(projects)

df = projects.copy()

df['days'] = (df['end_date']-df['start_date']).dt.days

df['business_days'] = df.apply(
    lambda r: len(pd.bdate_range(r['start_date'], r['end_date'])) - 1,
    axis=1
)

df


  project start_date   end_date
0   Alpha 2024-01-01 2024-01-10
1    Beta 2024-03-15 2024-03-20
2   Gamma 2024-12-23 2024-12-31


,project,start_date,end_date,days,business_days
0,Alpha,2024-01-01,2024-01-10,9,7
1,Beta,2024-03-15,2024-03-20,5,3
2,Gamma,2024-12-23,2024-12-31,8,6


**Concepts to use:**
1. `np.busday_count(start, end)` — counts business days between two dates.
2. `.apply(lambda row: ..., axis=1)` — row-wise custom function.
3. Convert Timestamps to `.date()` or `str` before passing to `np.busday_count`.

In [ ]:
# Optimised Solution
def add_business_days(df):
    df = df.copy()
    df['business_days'] = df.apply(
        lambda r: np.busday_count(r['start_date'].date(), r['end_date'].date()),
        axis=1
    )
    return df

print(add_business_days(projects))


---
## Q4 — Find Max & Min Without Built-ins (Easy)

**Problem:** Given a list of integers (including negatives and duplicates), find the maximum and minimum values without using `max()`, `min()`, or any sorting functions.

**Sample Input:**

| index | value |
|-------|-------|
| 0     | 3     |
| 1     | -1    |
| 2     | 7     |
| 3     | 0     |
| 4     | 7     |
| 5     | -9    |
| 6     | 4     |

**Sample Output:**

| result | value |
|--------|-------|
| max    | 7     |
| min    | -9    |

> Edge cases: `[5]` → max=5 min=5 · `[-3,-1,-9]` → max=-1 min=-9 · `[4,4,4]` → max=4 min=4

In [29]:
# Test inputs — cover edge cases
inputs = [
    [3, -1, 7, 0, 7, -9, 4],   # normal case
    [5],                        # single element
    [-3, -1, -9, -2],           # all negatives
    [4, 4, 4],                  # all duplicates
]


**Concepts to use:**
1. Initialise `current_max` and `current_min` with `nums[0]`.
2. Iterate and update with a simple `if` comparison.
3. Handle empty list as a guard clause.

In [30]:
# Optimised Solution
def find_max_min(nums):
    if not nums:
        return None, None
    current_max = current_min = nums[0]
    for n in nums[1:]:
        if n > current_max:
            current_max = n
        if n < current_min:
            current_min = n
    return current_max, current_min

for inp in inputs:
    print(f'Input: {inp} → max={find_max_min(inp)[0]}, min={find_max_min(inp)[1]}')


Input: [3, -1, 7, 0, 7, -9, 4] → max=7, min=-9
Input: [5] → max=5, min=5
Input: [-3, -1, -9, -2] → max=-1, min=-9
Input: [4, 4, 4] → max=4, min=4


---
## Q5 — Min-Max Normalisation (Easy)

**Problem:** Normalise the `score` column to the [0, 1] range using min-max scaling. Handle the edge case where all values are identical (division by zero).

**Sample Input:**

| student | score |
|---------|-------|
| A       | 50    |
| B       | 80    |
| C       | 20    |
| D       | 100   |
| E       | 50    |

**Sample Output:**

| student | score | normalised_score |
|---------|-------|-----------------|
| A       | 50    | 0.375           |
| B       | 80    | 0.750           |
| C       | 20    | 0.000           |
| D       | 100   | 1.000           |
| E       | 50    | 0.375           |

> Edge case — all same values `[5, 5]` → normalised_score = 0.0 for all rows

In [31]:
import pandas as pd

scores_df = pd.DataFrame({
    'student': ['A', 'B', 'C', 'D', 'E'],
    'score':   [50, 80, 20, 100, 50]
})
# Edge case: all same
scores_uniform = pd.DataFrame({'student': ['X','Y'], 'score': [5, 5]})
print(scores_df)


  student  score
0       A     50
1       B     80
2       C     20
3       D    100
4       E     50


**Concepts to use:**
1. Formula: `(x - min) / (max - min)`.
2. Guard against `max == min` (returns all 0s or NaN otherwise).
3. Vectorised pandas operations — no loops needed.

In [32]:
# Optimised Solution
def min_max_normalise(df, col='score'):
    df = df.copy()
    col_min, col_max = df[col].min(), df[col].max()
    if col_max == col_min:
        df['normalised_score'] = 0.0
    else:
        df['normalised_score'] = (df[col] - col_min) / (col_max - col_min)
    return df

print(min_max_normalise(scores_df))
print(min_max_normalise(scores_uniform))


  student  score  normalised_score
0       A     50             0.375
1       B     80             0.750
2       C     20             0.000
3       D    100             1.000
4       E     50             0.375
  student  score  normalised_score
0       X      5               0.0
1       Y      5               0.0


---
## Q6 — Impute Median by Group (Easy)

**Problem:** Fill missing `salary` values with the **median salary of the same department**. If an entire department has no salary, leave as NaN.

**Sample Input:**

| emp_id | dept  | salary |
|--------|-------|--------|
| 1      | Eng   | 90000  |
| 2      | Eng   | NaN    |
| 3      | Eng   | 80000  |
| 4      | HR    | NaN    |
| 5      | HR    | 60000  |
| 6      | Legal | NaN    |

**Sample Output:**

| emp_id | dept  | salary |
|--------|-------|--------|
| 1      | Eng   | 90000  |
| 2      | Eng   | 85000  |
| 3      | Eng   | 80000  |
| 4      | HR    | 60000  |
| 5      | HR    | 60000  |
| 6      | Legal | NaN    |

> Eng median = (80k+90k)/2 = 85k · HR median = 60k · Legal stays NaN (no data)

In [ ]:
import pandas as pd
import numpy as np

emp = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5, 6],
    'dept':   ['Eng', 'Eng', 'Eng', 'HR', 'HR', 'Legal'],
    'salary': [90000, np.nan, 80000, np.nan, 60000, np.nan]
})
print(emp)

df = emp.copy()

# df['salary'] = df['salary'].fillna(df['salary'].median())

grouped_Data = df.groupby('dept')['salary'].transform('median')

df['salary'] = df['salary'].fillna(df.groupby('dept')['salary'].transform('median'))

# df 
# df , type(df)
grouped_Data


   emp_id   dept   salary
0       1    Eng  90000.0
1       2    Eng      NaN
2       3    Eng  80000.0
3       4     HR      NaN
4       5     HR  60000.0
5       6  Legal      NaN


0    85000.0
1    85000.0
2    85000.0
3    60000.0
4    60000.0
5        NaN
Name: salary, dtype: float64

**Concepts to use:**
1. `groupby('dept')['salary'].transform('median')` — broadcasts group median back to each row.
2. `fillna(group_median)` — fills only the NaN positions.
3. `transform` is the key: it keeps the index aligned, unlike `agg`.

In [34]:
# Optimised Solution
def impute_median_by_group(df):
    df = df.copy()
    group_median = df.groupby('dept')['salary'].transform('median')
    df['salary'] = df['salary'].fillna(group_median)
    return df

print(impute_median_by_group(emp))


   emp_id   dept   salary
0       1    Eng  90000.0
1       2    Eng  85000.0
2       3    Eng  80000.0
3       4     HR  60000.0
4       5     HR  60000.0
5       6  Legal      NaN


---
## Q7 — Repeated Category Purchase (Easy)

**Problem:** Find all customers who made **more than one purchase in the same product category** on the same day. Return their `customer_id` and `category`.

**Sample Input:**

| order_id | customer_id | category    | purchase_date |
|----------|-------------|-------------|---------------|
| 1        | C1          | Electronics | 2024-01-05    |
| 2        | C1          | Electronics | 2024-01-05    |
| 3        | C1          | Clothing    | 2024-01-05    |
| 4        | C2          | Electronics | 2024-01-05    |
| 5        | C2          | Electronics | 2024-01-06    |

**Sample Output:**

| customer_id | category    | purchase_count |
|-------------|-------------|----------------|
| C1          | Electronics | 2              |

> C1 bought Electronics twice on same day ✅ · C2 bought on different days ❌

In [66]:
import pandas as pd

orders = pd.DataFrame({
    'order_id':      [1, 2, 3, 4, 5],
    'customer_id':   ['C1', 'C1', 'C1', 'C2', 'C2'],
    'category':      ['Electronics', 'Electronics', 'Clothing', 'Electronics', 'Electronics'],
    'purchase_date': pd.to_datetime(['2024-01-05','2024-01-05','2024-01-05','2024-01-05','2024-01-06'])
})
print(orders)

df = orders.copy()
df_grouped = df.groupby(['customer_id','purchase_date'])['category'].size().reset_index(name='purchase')

df['purchase'] = df.groupby(['customer_id', 'purchase_date'])['category'].cumcount()
# df   # ✅ All original columns preserved + the new cumcount column

df_grouped = df_grouped[df_grouped['purchase'] >= 3]

df_grouped


   order_id customer_id     category purchase_date
0         1          C1  Electronics    2024-01-05
1         2          C1  Electronics    2024-01-05
2         3          C1     Clothing    2024-01-05
3         4          C2  Electronics    2024-01-05
4         5          C2  Electronics    2024-01-06


,customer_id,purchase_date,purchase
0,C1,2024-01-05,3


**Concepts to use:**
1. `groupby(['customer_id','category','purchase_date']).size()` — count purchases per group.
2. Filter where `size > 1` to find repeated purchases.
3. `.reset_index()` to return a clean DataFrame.

In [ ]:
# Optimised Solution
def repeated_category_purchase(df):
    counts = (
        df.groupby(['customer_id', 'category', 'purchase_date'])
        .size()
        .reset_index(name='purchase_count')
    )
    return counts[counts['purchase_count'] > 1][['customer_id', 'category']]

print(repeated_category_purchase(orders))


---
## Q8 — Average Ride Duration with Timedelta (Easy)

**Problem:** Given a ride-sharing table, compute the **average ride duration in minutes** per city. Exclude rides where `end_time` is before `start_time` (data errors).

**Sample Input:**

| ride_id | city | start_time          | end_time            |
|---------|------|---------------------|---------------------|
| 1       | NYC  | 2024-01-01 08:00:00 | 2024-01-01 08:30:00 |
| 2       | NYC  | 2024-01-01 09:00:00 | 2024-01-01 08:50:00 |
| 3       | LA   | 2024-01-01 10:00:00 | 2024-01-01 10:45:00 |
| 4       | LA   | 2024-01-01 11:00:00 | 2024-01-01 11:20:00 |
| 5       | NYC  | 2024-01-01 12:00:00 | 2024-01-01 12:40:00 |

**Sample Output:**

| city | duration_mins |
|------|---------------|
| LA   | 32.5          |
| NYC  | 35.0          |

> Ride 2 dropped — end_time < start_time. NYC avg = (30+40)/2 = 35 · LA avg = (45+20)/2 = 32.5

In [74]:
import pandas as pd

rides = pd.DataFrame({
    'ride_id':    [1, 2, 3, 4, 5],
    'city':       ['NYC', 'NYC', 'LA', 'LA', 'NYC'],
    'start_time': pd.to_datetime([
        '2024-01-01 08:00', '2024-01-01 09:00',
        '2024-01-01 10:00', '2024-01-01 11:00', '2024-01-01 12:00'
    ]),
    'end_time': pd.to_datetime([
        '2024-01-01 08:30', '2024-01-01 08:50',   # ride 2 end < start
        '2024-01-01 10:45', '2024-01-01 11:20', '2024-01-01 12:40'
    ])
})
print(rides)

df = rides.copy()

df = df[df['end_time'] > df['start_time']]

df['duration_mins'] = (df['end_time'] - df['start_time']).dt.total_seconds()/60

df[['city','duration_mins']]
# df_grouped = df.groupby('city')



   ride_id city          start_time            end_time
0        1  NYC 2024-01-01 08:00:00 2024-01-01 08:30:00
1        2  NYC 2024-01-01 09:00:00 2024-01-01 08:50:00
2        3   LA 2024-01-01 10:00:00 2024-01-01 10:45:00
3        4   LA 2024-01-01 11:00:00 2024-01-01 11:20:00
4        5  NYC 2024-01-01 12:00:00 2024-01-01 12:40:00


,city,duration_mins
0,NYC,30.0
2,LA,45.0
3,LA,20.0
4,NYC,40.0


**Concepts to use:**
1. Subtract timestamps: `df['end_time'] - df['start_time']` gives a `Timedelta` Series.
2. `.dt.total_seconds() / 60` converts to minutes.
3. Filter rows where duration > 0 to drop erroneous records.

In [ ]:
# Optimised Solution
def avg_ride_duration(df):
    df = df.copy()
    df['duration_mins'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60
    df = df[df['duration_mins'] > 0]   # drop erroneous rides
    return df.groupby('city')['duration_mins'].mean().reset_index()

print(avg_ride_duration(rides))


---
## Q9 — IQR Outlier Removal (Easy → Medium)

**Problem:** Remove statistical outliers from a `salary` column using the **IQR method**. An outlier is any value below Q1 − 1.5×IQR or above Q3 + 1.5×IQR.

**Sample Input:**

| emp_id | dept | salary |
|--------|------|--------|
| 1      | Eng  | 40000  |
| 2      | Eng  | 45000  |
| 3      | Eng  | 50000  |
| 4      | HR   | 52000  |
| 5      | HR   | 55000  |
| 6      | HR   | 200000 |
| 7      | IT   | 10000  |

**Sample Output:**

| emp_id | dept | salary |
|--------|------|--------|
| 1      | Eng  | 40000  |
| 2      | Eng  | 45000  |
| 3      | Eng  | 50000  |
| 4      | HR   | 52000  |
| 5      | HR   | 55000  |

> 200000 (too high) and 10000 (too low) removed as outliers.

In [ ]:
import pandas as pd
import numpy as np

sal = pd.DataFrame({
    'emp_id': range(1, 8),
    'dept':   ['Eng','Eng','Eng','HR','HR','HR','IT'],
    'salary': [40000, 45000, 50000, 52000, 55000, 200000, 10000]
})
print(sal)


**Concepts to use:**
1. `Series.quantile(0.25)` and `quantile(0.75)` for Q1/Q3.
2. `IQR = Q3 - Q1`; bounds = `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.
3. Boolean mask to keep only inliers.

In [ ]:
# Optimised Solution
def remove_outliers_iqr(df, col='salary'):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return df[df[col].between(lower, upper)].copy()

print(remove_outliers_iqr(sal))


---
## Q10 — Forward Fill (ffill) within Groups (Medium)

**Problem:** A sensor DataFrame has missing `temperature` readings. Forward-fill missing values **within each device separately**. Do not propagate a value from one device into another.

**Sample Input:**

| device | timestamp  | temperature |
|--------|------------|-------------|
| A      | 2024-01-01 | 20.0        |
| A      | 2024-01-02 | NaN         |
| A      | 2024-01-03 | 22.0        |
| B      | 2024-01-01 | NaN         |
| B      | 2024-01-02 | 30.0        |
| B      | 2024-01-03 | NaN         |

**Sample Output:**

| device | timestamp  | temperature |
|--------|------------|-------------|
| A      | 2024-01-01 | 20.0        |
| A      | 2024-01-02 | 20.0        |
| A      | 2024-01-03 | 22.0        |
| B      | 2024-01-01 | NaN         |
| B      | 2024-01-02 | 30.0        |
| B      | 2024-01-03 | 30.0        |

> B row 1 stays NaN — no prior value to propagate within its group.

In [79]:
import pandas as pd
import numpy as np

sensors = pd.DataFrame({
    'device':      ['A','A','A','B','B','B'],
    'timestamp':   pd.to_datetime(['2024-01-01','2024-01-02','2024-01-03',
                                   '2024-01-01','2024-01-02','2024-01-03']),
    'temperature': [20.0, np.nan, 22.0, np.nan, 30.0, np.nan]
})
print(sensors)

df = sensors.copy()

df = df.groupby('device')['temperature'].ffill()

df


  device  timestamp  temperature
0      A 2024-01-01         20.0
1      A 2024-01-02          NaN
2      A 2024-01-03         22.0
3      B 2024-01-01          NaN
4      B 2024-01-02         30.0
5      B 2024-01-03          NaN


0    20.0
1    20.0
2    22.0
3     NaN
4    30.0
5    30.0
Name: temperature, dtype: float64

**Concepts to use:**
1. Sort by `['device','timestamp']` first to ensure correct chronological order.
2. `groupby('device')['temperature'].ffill()` — forward-fills within each group boundary.
3. Leading NaNs (no prior value in the group) remain NaN — correct behaviour.

In [ ]:
# Optimised Solution
def ffill_by_device(df):
    df = df.sort_values(['device', 'timestamp']).copy()
    df['temperature'] = df.groupby('device')['temperature'].ffill()
    return df

print(ffill_by_device(sensors))
